In [1]:
# ============================================================
# FER2013 - EfficientNet-B0 Benchmark Experiment
# ============================================================

import os
import time
import copy
import random
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

from torch.cuda.amp import autocast, GradScaler

from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from torchvision.datasets import ImageFolder

from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================

MODEL_NAME = "EfficientNetB0"

TRAIN_DIR = "/kaggle/input/datasets/ananthu017/emotion-detection-fer/train"
TEST_DIR = "/kaggle/input/datasets/ananthu017/emotion-detection-fer/test"

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 7
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
NUM_FOLDS = 5
RANDOM_SEED = 42

HEAD_ONLY_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUTPUT_DIR = f"./{MODEL_NAME}_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(RANDOM_SEED)

# ============================================================
# TRANSFORMS
# ============================================================

train_transform = transforms.Compose([

    transforms.Grayscale(num_output_channels=3),

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([

    transforms.Grayscale(num_output_channels=3),

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ============================================================
# DATASET
# ============================================================

class FERDataset(Dataset):

    def __init__(self, root_dir, transform=None):

        self.dataset = ImageFolder(root=root_dir)

        self.transform = transform

    def __len__(self):

        return len(self.dataset)

    def __getitem__(self, idx):

        image, label = self.dataset[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

# ============================================================
# LOAD DATASETS
# ============================================================

full_train_dataset = FERDataset(
    root_dir=TRAIN_DIR,
    transform=None
)

test_dataset = FERDataset(
    root_dir=TEST_DIR,
    transform=test_transform
)

class_names = full_train_dataset.dataset.classes

# ============================================================
# TARGETS + CLASS WEIGHTS
# ============================================================

targets = [label for _, label in full_train_dataset.dataset.samples]

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(targets),
    y=targets
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float
).to(DEVICE)

# ============================================================
# DATASET WRAPPER
# ============================================================

class TransformSubset(Dataset):

    def __init__(self, subset, transform=None):

        self.subset = subset
        self.transform = transform

    def __len__(self):

        return len(self.subset)

    def __getitem__(self, idx):

        image, label = self.subset[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

# ============================================================
# MODEL
# ============================================================

class EfficientNetB0FER(nn.Module):

    def __init__(self, num_classes=7):

        super(EfficientNetB0FER, self).__init__()

        self.backbone = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1
        )

        feature_dim = self.backbone.classifier[1].in_features

        # Replace classifier head
        self.backbone.classifier = nn.Sequential(

            nn.Dropout(0.4),

            nn.Linear(feature_dim, 256),

            nn.ReLU(inplace=True),

            nn.Dropout(0.3),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):

        return self.backbone(x)

# ============================================================
# FREEZING STRATEGY
# ============================================================

def freeze_for_phase1(model):

    # Freeze all feature extractor layers
    for param in model.backbone.features.parameters():
        param.requires_grad = False

    # Train classifier only
    for param in model.backbone.classifier.parameters():
        param.requires_grad = True

def unfreeze_for_phase2(model):

    # Freeze first 3 feature blocks
    for block in model.backbone.features[:3]:

        for param in block.parameters():
            param.requires_grad = False

    # Unfreeze remaining feature blocks
    for block in model.backbone.features[3:]:

        for param in block.parameters():
            param.requires_grad = True

    # Train classifier
    for param in model.backbone.classifier.parameters():
        param.requires_grad = True

# ============================================================
# METRICS
# ============================================================

def compute_metrics(y_true, y_pred):

    return {

        "accuracy": accuracy_score(y_true, y_pred),

        "precision": precision_score(
            y_true,
            y_pred,
            average='weighted',
            zero_division=0
        ),

        "recall": recall_score(
            y_true,
            y_pred,
            average='weighted',
            zero_division=0
        ),

        "weighted_f1": f1_score(
            y_true,
            y_pred,
            average='weighted',
            zero_division=0
        ),

        "macro_f1": f1_score(
            y_true,
            y_pred,
            average='macro',
            zero_division=0
        ),

        "per_class_f1": f1_score(
            y_true,
            y_pred,
            average=None,
            zero_division=0
        )
    }

# ============================================================
# PARAMETER COUNT
# ============================================================

def count_parameters(model):

    total_params = sum(p.numel() for p in model.parameters())

    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return total_params, trainable_params

# ============================================================
# TRAIN FUNCTION
# ============================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler
):

    model.train()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    for images, labels in tqdm(loader, leave=False):

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        with autocast(enabled=torch.cuda.is_available()):

            outputs = model(images)

            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)

    accuracy = accuracy_score(all_labels, all_preds)

    return epoch_loss, accuracy

# ============================================================
# VALIDATION FUNCTION
# ============================================================

def validate(model, loader, criterion):

    model.eval()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in tqdm(loader, leave=False):

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            with autocast(enabled=torch.cuda.is_available()):

                outputs = model(images)

                loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)

    accuracy = accuracy_score(all_labels, all_preds)

    metrics = compute_metrics(all_labels, all_preds)

    return epoch_loss, accuracy, metrics, all_labels, all_preds

# ============================================================
# CROSS VALIDATION
# ============================================================

print("\n================================================")
print("Starting 5-Fold Stratified Cross Validation")
print("================================================\n")

fold_results = []

start_training_time = time.time()

skf = StratifiedKFold(
    n_splits=NUM_FOLDS,
    shuffle=True,
    random_state=RANDOM_SEED
)

for fold, (train_idx, val_idx) in enumerate(
    skf.split(np.arange(len(targets)), targets)
):

    print(f"\n================ Fold {fold+1}/{NUM_FOLDS} ================\n")

    train_subset = Subset(full_train_dataset.dataset, train_idx)
    val_subset = Subset(full_train_dataset.dataset, val_idx)

    train_dataset = TransformSubset(
        train_subset,
        transform=train_transform
    )

    val_dataset = TransformSubset(
        val_subset,
        transform=test_transform
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    model = EfficientNetB0FER(
        num_classes=NUM_CLASSES
    ).to(DEVICE)

    freeze_for_phase1(model)

    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )

    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=3
    )

    scaler = GradScaler()

    best_val_loss = np.inf
    best_model_wts = copy.deepcopy(model.state_dict())

    early_stop_counter = 0

    history = []

    for epoch in range(NUM_EPOCHS):

        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]")

        # ====================================================
        # PHASE 2
        # ====================================================

        if epoch == HEAD_ONLY_EPOCHS:

            print("\nUnfreezing EfficientNet deeper layers...\n")

            unfreeze_for_phase2(model)

            optimizer = optim.Adam(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=LEARNING_RATE
            )

        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scaler
        )

        val_loss, val_acc, val_metrics, _, _ = validate(
            model,
            val_loader,
            criterion
        )

        scheduler.step(val_loss)

        history.append({

            "epoch": epoch + 1,

            "train_loss": train_loss,

            "val_loss": val_loss,

            "train_accuracy": train_acc,

            "val_accuracy": val_acc
        })

        print(
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Accuracy: {val_acc:.4f} | "
            f"Weighted F1: {val_metrics['weighted_f1']:.4f}"
        )

        # ====================================================
        # SAVE BEST MODEL
        # ====================================================

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_model_wts = copy.deepcopy(model.state_dict())

            torch.save(
                model.state_dict(),
                os.path.join(
                    OUTPUT_DIR,
                    f"{MODEL_NAME}_fold{fold+1}_best.pth"
                )
            )

            early_stop_counter = 0

        else:
            early_stop_counter += 1

        # ====================================================
        # EARLY STOPPING
        # ====================================================

        if early_stop_counter >= EARLY_STOPPING_PATIENCE:

            print("\nEarly stopping triggered.\n")

            break

    # ========================================================
    # LOAD BEST MODEL
    # ========================================================

    model.load_state_dict(best_model_wts)

    val_loss, val_acc, val_metrics, _, _ = validate(
        model,
        val_loader,
        criterion
    )

    fold_results.append({

        "accuracy": val_metrics["accuracy"],

        "weighted_f1": val_metrics["weighted_f1"],

        "macro_f1": val_metrics["macro_f1"]
    })

    # ========================================================
    # SAVE TRAINING LOG
    # ========================================================

    history_df = pd.DataFrame(history)

    history_df.to_csv(
        os.path.join(
            OUTPUT_DIR,
            f"{MODEL_NAME}_fold{fold+1}_training_log.csv"
        ),
        index=False
    )

# ============================================================
# CROSS VALIDATION SUMMARY
# ============================================================

cv_accuracies = [x["accuracy"] for x in fold_results]
cv_weighted_f1 = [x["weighted_f1"] for x in fold_results]
cv_macro_f1 = [x["macro_f1"] for x in fold_results]

mean_acc = np.mean(cv_accuracies)
std_acc = np.std(cv_accuracies)

mean_weighted_f1 = np.mean(cv_weighted_f1)
std_weighted_f1 = np.std(cv_weighted_f1)

mean_macro_f1 = np.mean(cv_macro_f1)
std_macro_f1 = np.std(cv_macro_f1)

# ============================================================
# FINAL TRAINING
# ============================================================

print("\n================================================")
print("Training Final Model on Full Training Dataset")
print("================================================\n")

final_train_dataset = FERDataset(
    root_dir=TRAIN_DIR,
    transform=train_transform
)

final_train_loader = DataLoader(
    final_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

final_model = EfficientNetB0FER(
    num_classes=NUM_CLASSES
).to(DEVICE)

freeze_for_phase1(final_model)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, final_model.parameters()),
    lr=LEARNING_RATE
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3
)

scaler = GradScaler()

best_model_wts = copy.deepcopy(final_model.state_dict())
best_loss = np.inf

history = []

for epoch in range(NUM_EPOCHS):

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}]")

    if epoch == HEAD_ONLY_EPOCHS:

        print("\nUnfreezing EfficientNet deeper layers...\n")

        unfreeze_for_phase2(final_model)

        optimizer = optim.Adam(
            filter(lambda p: p.requires_grad, final_model.parameters()),
            lr=LEARNING_RATE
        )

    train_loss, train_acc = train_one_epoch(
        final_model,
        final_train_loader,
        criterion,
        optimizer,
        scaler
    )

    scheduler.step(train_loss)

    history.append({

        "epoch": epoch + 1,

        "train_loss": train_loss,

        "train_accuracy": train_acc
    })

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Train Accuracy: {train_acc:.4f}"
    )

    if train_loss < best_loss:

        best_loss = train_loss

        best_model_wts = copy.deepcopy(final_model.state_dict())

        torch.save(
            final_model.state_dict(),
            os.path.join(
                OUTPUT_DIR,
                f"{MODEL_NAME}_final_best.pth"
            )
        )

final_model.load_state_dict(best_model_wts)

# ============================================================
# TEST EVALUATION
# ============================================================

print("\n================================================")
print("Final Evaluation on Test Set")
print("================================================\n")

test_loss, test_acc, test_metrics, y_true, y_pred = validate(
    final_model,
    test_loader,
    criterion
)

# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(y_true, y_pred)

cm_normalized = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted")
plt.ylabel("True")

plt.title(
    f"{MODEL_NAME} - Normalized Confusion Matrix"
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        f"{MODEL_NAME}_confusion_matrix.png"
    )
)

plt.close()

# ============================================================
# PER-CLASS F1 SCORE
# ============================================================

per_class_f1 = test_metrics["per_class_f1"]

plt.figure(figsize=(10, 6))

sns.barplot(
    x=class_names,
    y=per_class_f1
)

plt.ylim(0, 1)

plt.title(f"{MODEL_NAME} - Per-Class F1 Score")

plt.ylabel("F1 Score")

plt.tight_layout()

plt.savefig(
    os.path.join(
        OUTPUT_DIR,
        f"{MODEL_NAME}_per_class_f1.png"
    )
)

plt.close()

# ============================================================
# SAVE TRAINING LOG
# ============================================================

history_df = pd.DataFrame(history)

history_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        f"{MODEL_NAME}_final_training_log.csv"
    ),
    index=False
)

# ============================================================
# INFERENCE TIME
# ============================================================

dummy_input = torch.randn(
    1,
    3,
    IMAGE_SIZE,
    IMAGE_SIZE
).to(DEVICE)

final_model.eval()

num_runs = 100

starter = time.time()

with torch.no_grad():

    for _ in range(num_runs):

        _ = final_model(dummy_input)

ender = time.time()

avg_inference_time = (
    (ender - starter) / num_runs
)

# ============================================================
# TOTAL TRAINING TIME
# ============================================================

total_training_time = time.time() - start_training_time

# ============================================================
# PARAMETER COUNT
# ============================================================

total_params, trainable_params = count_parameters(
    final_model
)

# ============================================================
# FINAL RESULTS
# ============================================================

print("\n================================================")
print("FINAL RESULTS")
print("================================================\n")

print(f"Mean CV Accuracy      : {mean_acc:.4f}")
print(f"Std CV Accuracy       : {std_acc:.4f}")

print(f"\nMean Weighted F1      : {mean_weighted_f1:.4f}")
print(f"Std Weighted F1       : {std_weighted_f1:.4f}")

print(f"\nMean Macro F1         : {mean_macro_f1:.4f}")
print(f"Std Macro F1          : {std_macro_f1:.4f}")

print("\n------------------------------------------------")

print(f"Final Test Accuracy   : {test_metrics['accuracy']:.4f}")

print(f"Final Weighted F1     : {test_metrics['weighted_f1']:.4f}")

print(f"Final Macro F1        : {test_metrics['macro_f1']:.4f}")

print("\n------------------------------------------------")

print(f"Total Parameters      : {total_params:,}")

print(f"Trainable Parameters  : {trainable_params:,}")

print(
    f"\nAvg Inference Time    : "
    f"{avg_inference_time:.6f} sec/image"
)

print(
    f"\nTotal Training Time   : "
    f"{total_training_time/60:.2f} minutes"
)

print("\n================================================")
print("Classification Report")
print("================================================\n")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4
    )
)

print("\n================================================")
print("Experiment Completed Successfully")
print("================================================")


Starting 5-Fold Stratified Cross Validation


================ Fold 1/5 ================

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 131MB/s] 


Epoch [1/30]


Train Loss: 1.8408 | Val Loss: 1.7549 | Val Accuracy: 0.3520 | Weighted F1: 0.3195
Epoch [2/30]


Train Loss: 1.7351 | Val Loss: 1.7008 | Val Accuracy: 0.3736 | Weighted F1: 0.3693
Epoch [3/30]


...

Train Loss: 0.2393 | Train Accuracy: 0.9070
Epoch [28/30]


Train Loss: 0.2214 | Train Accuracy: 0.9134
Epoch [29/30]


Train Loss: 0.2169 | Train Accuracy: 0.9181
Epoch [30/30]


Train Loss: 0.2036 | Train Accuracy: 0.9223

Final Evaluation on Test Set




FINAL RESULTS

Mean CV Accuracy      : 0.6658
Std CV Accuracy       : 0.0035

Mean Weighted F1      : 0.6637
Std Weighted F1       : 0.0029

Mean Macro F1         : 0.6386
Std Macro F1          : 0.0023

------------------------------------------------
Final Test Accuracy   : 0.7019
Final Weighted F1     : 0.7023
Final Macro F1        : 0.6939

------------------------------------------------
Total Parameters      : 4,337,283
Trainable Parameters  : 4,318,193

Avg Inference Time    : 0.009338 sec/image

Total Training Time   : 149.86 minutes

Classification Report

              precision    recall  f1-score   support

       angry     0.6313    0.6148    0.6230       958
   disgusted     0.7383    0.7117    0.7248       111
     fearful     0.5819    0.5273    0.5533      1024
       happy     0.8841    0.8732    0.8786      1774
     neutral     0.6436    0.6926    0.6672      1233
         sad     0.5658    0.6071    0.5857      1247
   surprised     0.8449    0.8063    0.8251     